# Semana 5 – Actividad 5: Clasificación multiclase (MLP, TensorFlow Keras)

**Curso:** Deep Learning - Conceptos (601539) · FU / CAD2202023205 / EIAIPA2026 · Semana 5

Red **fully connected** (sin convolución) sobre el dataset **Digits** de scikit-learn (10 clases). Se comparan **optimizadores** y el efecto de **tasa de aprendizaje**, **tamaño de lote** y **épocas**, con la misma partición de datos y semillas para que los resultados sean **reproducibles** y comparables.

**Entrega:** ejecutar en **Google Colab**; en GitHub, carpeta **`week5/`** con este `.ipynb` y el **`README.md`** de soporte al mismo nivel.


In [3]:
import subprocess
import sys


def pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", *args], check=False)


# Quitar restos de intentos fallidos (ignora error si no estaba instalado)
pip("uninstall", "-y", "tensorflow", "tensorflow-intel")
pip("install", "-U", "pip", "-q")
r = pip(
    "install",
    "tensorflow",
    "scikit-learn",
    "matplotlib",
    "numpy",
    "pandas",
    "-q",
)
if r.returncode != 0:
    raise RuntimeError(
        "Falló pip install tensorflow. En Windows suele deberse a rutas largas: "
        "activa Long Path Support o usa un venv en C:\\... corto, o ejecuta en Google Colab."
    )
print("Dependencias instaladas. Reinicia el kernel si Jupyter lo pide; luego ejecuta la siguiente celda.")


RuntimeError: Falló pip install tensorflow. En Windows suele deberse a rutas largas: activa Long Path Support o usa un venv en C:\... corto, o ejecuta en Google Colab.

## 1. Dataset y preparación

- **Digits:** 1797 muestras, 64 características (imagen 8×8 aplanada), 10 clases (dígitos 0–9).
- Partición **estratificada:** train / validación / test.
- **StandardScaler** ajustado solo sobre train.


In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

X, y = load_digits(return_X_y=True)
n_classes = len(np.unique(y))
n_features = X.shape[1]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train_full
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.astype(np.float32))
X_val = scaler.transform(X_val.astype(np.float32))
X_test = scaler.transform(X_test.astype(np.float32))

print(f"Features: {n_features} | Clases: {n_classes}")
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")


ModuleNotFoundError: No module named 'tensorflow.python'

## 2. Arquitectura MLP (multicategoría)

- Capas densas **ReLU** y salida **softmax** (10 neuronas).
- Pérdida: `sparse_categorical_crossentropy` (etiquetas enteras).
- Métrica: `accuracy`.

Función reutilizable `build_model(optimizer)` para que **solo cambie el optimizador** o los hiperparámetros que se estudien.


In [ ]:
def build_model(optimizer):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(n_features,)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def train_model(optimizer, epochs=80, batch_size=32, verbose=0):
    model = build_model(optimizer)
    hist = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose,
    )
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    return model, hist, test_loss, test_acc


## 3. Comparación de optimizadores

Mismas condiciones: **80 épocas**, **batch 32**, misma arquitectura y datos. Se usan tasas de aprendizaje **típicas** por optimizador (comentadas en código); el propósito es observar **curvas** y **accuracy en test**.


In [ ]:
EPOCHS_OPT = 80
BATCH_OPT = 32

runs_opt = []
runs_opt.append(("SGD (momentum=0.9)", tf.keras.optimizers.SGD(learning_rate=0.05, momentum=0.9)))
runs_opt.append(("Adam", tf.keras.optimizers.Adam(learning_rate=1e-3)))
runs_opt.append(("RMSprop", tf.keras.optimizers.RMSprop(learning_rate=1e-3)))

histories_opt = {}
test_results_opt = {}
for name, opt in runs_opt:
    _, h, te_loss, te_acc = train_model(opt, epochs=EPOCHS_OPT, batch_size=BATCH_OPT, verbose=0)
    histories_opt[name] = h.history
    test_results_opt[name] = {"test_loss": float(te_loss), "test_accuracy": float(te_acc)}
    print(f"{name}: test accuracy = {te_acc:.4f}, test loss = {te_loss:.4f}")


## 4. Hiperparámetro: tasa de aprendizaje (Adam)

Fijamos **Adam** y **batch 32**, **80 épocas**. Se prueban tres valores de `learning_rate`.


In [ ]:
EPOCHS_LR = 80
BATCH_LR = 32
lr_list = [1e-2, 1e-3, 1e-4]
histories_lr = {}
test_results_lr = {}
for lr in lr_list:
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    _, h, te_loss, te_acc = train_model(opt, epochs=EPOCHS_LR, batch_size=BATCH_LR, verbose=0)
    key = f"Adam lr={lr}"
    histories_lr[key] = h.history
    test_results_lr[key] = {"test_loss": float(te_loss), "test_accuracy": float(te_acc)}
    print(f"{key}: test accuracy = {te_acc:.4f}")


## 5. Hiperparámetro: tamaño de lote

Mismo **Adam (lr=1e-3)**, **80 épocas**. Batch pequeño implica más ruido por actualización; batch grande reduce el número de actualizaciones por época.


In [ ]:
EPOCHS_BS = 80
LR_BS = 1e-3
batch_sizes = [8, 32, 128]
histories_bs = {}
test_results_bs = {}
for bs in batch_sizes:
    opt = tf.keras.optimizers.Adam(learning_rate=LR_BS)
    _, h, te_loss, te_acc = train_model(opt, epochs=EPOCHS_BS, batch_size=bs, verbose=0)
    key = f"batch={bs}"
    histories_bs[key] = h.history
    test_results_bs[key] = {"test_loss": float(te_loss), "test_accuracy": float(te_acc)}
    print(f"{key}: test accuracy = {te_acc:.4f}")


## 6. Gráficos y tabla resumen


In [ ]:
def plot_group(hdict, title_prefix):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for label, h in hdict.items():
        ax[0].plot(h["loss"], label=f"{label} (train)")
        ax[0].plot(h["val_loss"], linestyle="--", alpha=0.7)
    ax[0].set_title(f"{title_prefix} — loss")
    ax[0].set_xlabel("Época")
    ax[0].legend(fontsize=8)
    for label, h in hdict.items():
        ax[1].plot(h["accuracy"], label=f"{label} (train)")
        ax[1].plot(h["val_accuracy"], linestyle="--", alpha=0.7)
    ax[1].set_title(f"{title_prefix} — accuracy")
    ax[1].set_xlabel("Época")
    ax[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


plot_group(histories_opt, "Optimizadores")
plot_group(histories_lr, "Adam: learning rate")
plot_group(histories_bs, "Adam: batch size")

rows = []
for k, v in test_results_opt.items():
    rows.append({"experimento": "optimizador", "config": k, **v})
for k, v in test_results_lr.items():
    rows.append({"experimento": "lr", "config": k, **v})
for k, v in test_results_bs.items():
    rows.append({"experimento": "batch", "config": k, **v})

df = pd.DataFrame(rows)
display(df.sort_values(["experimento", "test_accuracy"], ascending=[True, False]))


## 7. Análisis y conclusiones breves

### Estabilidad

En la comparación de optimizadores, **Adam** mostró el comportamiento más estable y el mejor rendimiento global en test (`accuracy=0.9861`, `loss=0.1066`). **SGD con momentum** también fue consistente (`accuracy=0.9778`), aunque con pérdida final mayor que Adam. **RMSprop** se mantuvo competitivo pero ligeramente por debajo de Adam y SGD. En el barrido de tasa de aprendizaje, `lr=0.01` presentó inestabilidad (picos en `loss`) y peor desempeño final (`accuracy=0.9611`, `loss=0.8194`).

### Velocidad de convergencia

Las curvas muestran que Adam y SGD alcanzan rápidamente zonas de alta `accuracy`, pero Adam converge antes a valores bajos de `loss`. Con `Adam lr=0.001` y `lr=0.0001` la convergencia fue más suave; sin embargo, `lr=0.0001` necesitó más épocas para acercarse al mejor rendimiento. Esto confirma el trade-off típico: tasas muy bajas son estables pero más lentas, y tasas altas aceleran al inicio pero pueden volverse inestables.

### Hallazgos al ajustar hiperparámetros y optimización

En este problema multiclase (Digits), la mejor configuración fue **Adam** con sus parámetros base del experimento. Para el learning rate, **`lr=0.001`** ofreció el mejor equilibrio entre estabilidad y rapidez. En tamaño de lote, **`batch=8`** obtuvo la mejor `test_accuracy` (`0.9778`), mientras que `batch=32` y `batch=128` quedaron en `0.9722`; aun así, `batch=32` logró menor pérdida (`0.1175`), mostrando que minimizar `loss` no siempre maximiza `accuracy`.

### Limitaciones

El modelo es un MLP sobre 64 características y no explota explícitamente la estructura espacial 8x8 de las imágenes. Como mejora futura, conviene comparar contra una CNN simple y validar si mantiene o supera el rendimiento observado.

---
*Evidencia reproducible: semillas fijas, hiperparámetros explícitos en código, curvas y tabla de resultados.*
